# Sample v2: 500 suspended / 500 active eligible accused accounts, activity before accusation

Same pool as `sample_1000_v1_stratified_eligible.ipynb` with two more rules, so every account has a positive
time from first activity to first accusation (needed for the AFT model):

**Eligible** = all of:
1. in `accused_accounts_not_deleted.parquet` (PRAW status active or suspended);
2. has an Arctic Shift history pull marked `complete` in `arctic_pull_done.parquet` (capped 10k-per-stream pulls count as complete and are flagged by `history_truncated`);
3. **first action after r/TheseFuckingAccounts existed**: the earliest of Arctic Shift's `earliest_post_at`, `earliest_comment_at` and the oldest stored post or comment is after 2014-12-21 00:03:16 UTC;
4. **first accused before 30 November 2022**: `first_mention_utc` < 2022-11-30 00:00:00 UTC;
5. **first action before 30 November 2022**: `first_action_utc` < 2022-11-30 00:00:00 UTC;
6. **first action before first accusation**: `first_action_utc` < `first_mention_utc`. Accounts failing this have no archived
   activity until after they were named — either batch-listed while dormant, activity removed before archiving, or named in a
   later edit of the accusing post (the CSV text includes edits but `first_mention_utc` is the post's creation time).

**Resampling:** the v1 accounts (`sample_1000_output/`) that pass all six rules are kept; the ones that fail are replaced by a
stratified draw (`random_state=2026`) from the eligible accounts not already in v1. The result is a simple random sample of
the new pool within each status.

Inputs are read-only; outputs go to `sample_1000_v2_output/` and `save_new()` refuses to overwrite.

In [ ]:
# Libraries.
import pandas as pd
import numpy as np
import os
import re
import glob
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

# Folders, sample size, random seed and the cut-off times (subreddit creation; 30 Nov 2022).
DATA_DIR = os.path.join('..', 'p2_data_cleaning', 'data')
OUT_DIR = 'sample_1000_v2_output'
PREV_DIR = 'sample_1000_output'
N_PER_GROUP = 500
SEED = 2026
TFA_CREATED_UTC = 1419120196                                                  # 2014-12-21 00:03:16 UTC, PRAW subreddit.created_utc
ACCUSED_BEFORE_UTC = int(pd.Timestamp('2022-11-30', tz='UTC').timestamp())   # 1669766400
os.makedirs(OUT_DIR, exist_ok=True)

# Write a parquet file only if it does not exist yet (never overwrite a saved sample).
def save_new(df, name):
    path = os.path.join(OUT_DIR, name)
    if os.path.exists(path):
        print(f'exists, not overwritten: {path}')
        return
    df.to_parquet(path, engine='pyarrow')
    print(f'wrote {path}  {df.shape}')

# Read the stored Arctic Shift history (kind = 'posts' or 'comments'), optionally only some columns/accounts.
def read_history(kind, columns=None, keys=None):
    # file by file: some parts store all-null columns as type null, which breaks a unified dataset schema.
    # repair-* parts re-pulled whole histories, so rows repeat across files -> dedupe on (outer_key, id)
    filters = [('outer_key', 'in', list(keys))] if keys is not None else None
    parts = [pd.read_parquet(f, engine='pyarrow', columns=columns, filters=filters)
             for f in sorted(glob.glob(os.path.join(DATA_DIR, f'arctic_{kind}', '*.parquet')))]
    df = pd.concat([p for p in parts if len(p)], ignore_index=True)
    return df.drop_duplicates(['outer_key', 'id']).reset_index(drop=True)

## 1. Eligible pool

In [ ]:
# Accused accounts (not deleted), the history-pull ledger, and Arctic Shift account metadata.
accused = pd.read_parquet(os.path.join(DATA_DIR, 'accused_accounts_not_deleted.parquet'))
pull_done = pd.read_parquet(os.path.join(DATA_DIR, 'arctic_pull_done.parquet'))
meta = pd.read_parquet(os.path.join(DATA_DIR, 'account_meta_arctic.parquet'))
assert pull_done['username_key'].is_unique
print('duplicate meta keys (kept last):', meta['username_key'].duplicated().sum())
meta = meta.drop_duplicates('username_key', keep='last')

# Earliest stored post or comment per account.
hist_first = pd.concat([read_history('posts', ['outer_key', 'id', 'created_utc']),
                        read_history('comments', ['outer_key', 'id', 'created_utc'])]
                       ).groupby('outer_key')['created_utc'].min()

# One row per accused account, with pull status and the archive's first-post/first-comment times.
pool = (accused.merge(pull_done, on='username_key', how='left', validate='one_to_one')
               .merge(meta[['username_key', 'earliest_post_at', 'earliest_comment_at']],
                      on='username_key', how='left', validate='one_to_one'))
pool['history_first_utc'] = pool['username_key'].map(hist_first)
# First action = earliest of the archive's first post, first comment, and first stored row.
pool['first_action_utc'] = pool[['earliest_post_at', 'earliest_comment_at', 'history_first_utc']].min(axis=1)
pool['complete'] = pool['complete'].eq(True)          # accounts with no ledger row are not complete

# Eligibility funnel: v1 rules plus two new ones (first action before 30 Nov 2022, and before first accusation).
steps = {'accused, not deleted': pd.Series(True, index=pool.index)}
steps['+ complete Arctic history'] = steps['accused, not deleted'] & pool['complete']
steps['+ first action after TFA created'] = steps['+ complete Arctic history'] & (pool['first_action_utc'] > TFA_CREATED_UTC)
steps['+ first accused before 2022-11-30'] = steps['+ first action after TFA created'] & (pool['first_mention_utc'] < ACCUSED_BEFORE_UTC)
steps['+ first action before 2022-11-30'] = steps['+ first accused before 2022-11-30'] & (pool['first_action_utc'] < ACCUSED_BEFORE_UTC)
steps['+ first action before first accused'] = steps['+ first action before 2022-11-30'] & (pool['first_action_utc'] < pool['first_mention_utc'])
funnel = pd.DataFrame({k: pool.loc[v, 'account_status'].value_counts() for k, v in steps.items()}).T
funnel['total'] = funnel.sum(axis=1)
funnel

In [ ]:
# Eligible pool = accounts that pass every rule. Shows how many histories are cut at the 10k-item cap,
# and median activity by status.
eligible = pool[steps['+ first action before first accused']].reset_index(drop=True)
# v2 only adds rules, so every v2-eligible account must be in the v1 pool.
prev_pool = pd.read_parquet(os.path.join(PREV_DIR, 'eligible_pool.parquet'))
assert eligible['username_key'].isin(prev_pool['username_key']).all(), 'new pool should be a subset of the v1 pool'
print('history truncated at the 10k cap:', eligible['history_truncated'].sum())
# Days from first action to first accusation.
eligible['days_to_accused'] = (eligible['first_mention_utc'] - eligible['first_action_utc']) / 86400
eligible.groupby('account_status')[['n_posts', 'n_comments', 'times_mentioned', 'days_to_accused']].median()

## 2. Draw the sample

In [ ]:
# Keep every v1 sampled account that is still eligible, and replace each dropped one
# with a new random draw of the same status from accounts not in the v1 sample.
prev_sample = pd.read_parquet(os.path.join(PREV_DIR, 'sample_accounts.parquet'), columns=['username_key', 'account_status'])
kept = prev_sample[prev_sample['username_key'].isin(eligible['username_key'])]
dropped = prev_sample[~prev_sample['username_key'].isin(eligible['username_key'])]
need = N_PER_GROUP - kept['account_status'].value_counts()
print('v1 accounts dropped:', dropped['account_status'].value_counts().to_dict())

candidates = eligible[~eligible['username_key'].isin(prev_sample['username_key'])]
replacements = (candidates.groupby('account_status', group_keys=False)
                          .apply(lambda g: g.sample(n=int(need[g.name]), random_state=SEED)))
new_keys = set(kept['username_key']) | set(replacements['username_key'])
sample = eligible[eligible['username_key'].isin(new_keys)].reset_index(drop=True)
# Checks: no duplicates; every account acted before it was accused, and inside the study window.
assert sample['username_key'].is_unique
assert (sample['first_action_utc'] < sample['first_mention_utc']).all()
assert (sample['first_action_utc'] > TFA_CREATED_UTC).all() and (sample['first_action_utc'] < ACCUSED_BEFORE_UTC).all()
print(sample['account_status'].value_counts().to_string())
print('kept from v1:', len(kept), '| replacements:', replacements['account_status'].value_counts().to_dict())

## 3. Attach Arctic Shift metadata and PRAW profiles

In [ ]:
# Attach Arctic Shift metadata (prefix arctic_) and PRAW profile fields (prefix praw_) to the sample.
# Image URL columns are dropped; account_status comes from the sample, not the profile table.
profiles = pd.read_parquet(os.path.join(DATA_DIR, 'account_profiles_full.parquet'))
meta_cols = (meta.drop(columns=['fetch_error', 'earliest_post_at', 'earliest_comment_at'])
                 .add_prefix('arctic_').rename(columns={'arctic_username_key': 'username_key'}))
profile_cols = (profiles.drop(columns=['account_status', 'icon_img', 'snoovatar_img'])
                        .add_prefix('praw_').rename(columns={'praw_username_key': 'username_key'}))
sample_accounts = (sample.merge(meta_cols, on='username_key', how='left', validate='one_to_one')
                         .merge(profile_cols, on='username_key', how='left', validate='one_to_one'))
print(sample_accounts.shape)

## 4. The sample's stored history

In [ ]:
# Load the stored posts and comments of the sampled accounts only.
keys = set(sample['username_key'])
sample_posts = read_history('posts', keys=keys)
sample_comments = read_history('comments', keys=keys)
print(f'posts {len(sample_posts):,}   comments {len(sample_comments):,}')

# Rows stored per account, by status: a check that every sampled account has history.
counts = pd.DataFrame({'posts': sample_posts['outer_key'].value_counts(),
                       'comments': sample_comments['outer_key'].value_counts()}).reindex(sorted(keys)).fillna(0).astype(int)
counts['account_status'] = counts.index.map(sample.set_index('username_key')['account_status'])
counts.groupby('account_status')[['posts', 'comments']].agg(['sum', 'median', 'mean', 'max'])

## 5. Save

In [ ]:
# Save the eligible pool, the sample accounts, and their posts and comments (never overwrites).
save_new(eligible[['username_key', 'account_status', 'first_mention_utc', 'first_action_utc', 'days_to_accused', 'history_truncated']],
         'eligible_pool.parquet')
save_new(sample_accounts, 'sample_accounts.parquet')
save_new(sample_posts, 'sample_posts.parquet')
save_new(sample_comments, 'sample_comments.parquet')
# v1 sampled accounts that were dropped, with the reason.
save_new(dropped.assign(reason='no archived activity before first accusation'), 'dropped_from_v1.parquet')